# Pharmasense — TFT Training
**Temporal Fusion Transformer for demand baseline forecasting**

Used for: stable disease baselines (non-communicable demand patterns, seasonal ADD/cholera without spatial spread component)

Library: `pytorch-forecasting` (wraps PyTorch Lightning + TFT architecture)

**Runtime:** GPU (T4). Go to Runtime → Change runtime type → T4 GPU before running.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abhi-dev99/agtechathon/blob/main/ml/notebooks/02_train_tft.ipynb)

## 0. Install dependencies

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'pytorch-forecasting', 'pytorch-lightning', 'psycopg2-binary',
    'pandas', 'numpy', 'scikit-learn', 'matplotlib'], check=True)

print('Dependencies installed.')

## 1. Config

In [ ]:
# ── Supabase connection ────────────────────────────────────────────────────────
DB_HOST     = 'db.xwtvujwdybvjbaucshbt.supabase.co'
DB_PORT     = 5432
DB_NAME     = 'postgres'
DB_USER     = 'postgres'
DB_PASSWORD = 'pharmasons99lite'

# ── TFT config ────────────────────────────────────────────────────────────────
DISEASES         = ['dengue', 'malaria', 'cholera', 'acute_diarrhoeal_disease']
MAX_ENCODER_LEN  = 24   # lookback: 24 weeks (~6 months)
MAX_PREDICTION_LEN = 4  # forecast: 4 weeks ahead
BATCH_SIZE       = 64
MAX_EPOCHS       = 50
LEARNING_RATE    = 1e-3
HIDDEN_SIZE      = 32
ATTENTION_HEADS  = 4
DROPOUT          = 0.1

DRIVE_OUTPUT = '/content/drive/MyDrive/pharmasense_models/'

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f'Drive ready: {DRIVE_OUTPUT}')

## 3. Load + prepare data

In [ ]:
import psycopg2
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

conn = psycopg2.connect(
    host=DB_HOST, port=DB_PORT, dbname=DB_NAME,
    user=DB_USER, password=DB_PASSWORD, sslmode='require'
)

# Load districts (for static covariates: state, lat, lon)
dist_df = pd.read_sql("""
    SELECT id, name, state, centroid_lat, centroid_lon FROM districts;
""", conn)

# Load epid data
epid_df = pd.read_sql("""
    SELECT district_id, disease, iso_week, week_start_date, cases, deaths
    FROM epid_weekly
    ORDER BY week_start_date, district_id;
""", conn)
conn.close()

epid_df['week_start_date'] = pd.to_datetime(epid_df['week_start_date'])
epid_df['log_cases']       = np.log1p(epid_df['cases'].astype(float))

# Merge in district static info
epid_df = epid_df.merge(dist_df[['id','state','centroid_lat','centroid_lon']],
                        left_on='district_id', right_on='id', how='left')

# ── Encode categoricals ───────────────────────────────────────────────────────
le_district = LabelEncoder()
le_state    = LabelEncoder()
le_disease  = LabelEncoder()

epid_df['district_enc'] = le_district.fit_transform(epid_df['district_id'].astype(str))
epid_df['state_enc']    = le_state.fit_transform(epid_df['state'].fillna('Unknown'))
epid_df['disease_enc']  = le_disease.fit_transform(epid_df['disease'])

# Time index (integer week number from start)
min_date            = epid_df['week_start_date'].min()
epid_df['time_idx'] = ((epid_df['week_start_date'] - min_date).dt.days // 7).astype(int)

# Create unique group key: district_id + disease
epid_df['group_id'] = epid_df['district_id'].astype(str) + '_' + epid_df['disease']

# Only keep groups with enough history (at least MAX_ENCODER_LEN + MAX_PREDICTION_LEN weeks)
min_len = MAX_ENCODER_LEN + MAX_PREDICTION_LEN
group_counts = epid_df.groupby('group_id')['time_idx'].count()
valid_groups = group_counts[group_counts >= min_len].index
epid_df = epid_df[epid_df['group_id'].isin(valid_groups)].copy()

print(f'Records after filtering: {len(epid_df):,}')
print(f'Valid (district, disease) groups: {epid_df["group_id"].nunique()}')
print(f'Time range: {epid_df["week_start_date"].min()} -> {epid_df["week_start_date"].max()}')

## 4. Build PyTorch Forecasting TimeSeriesDataSet

In [ ]:
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss
import pytorch_lightning as pl
from torch.utils.data import DataLoader

# Train/val split by time
max_time_idx  = epid_df['time_idx'].max()
train_cutoff  = int(max_time_idx * 0.80)

training_df = epid_df[epid_df['time_idx'] <= train_cutoff]
val_df      = epid_df

training = TimeSeriesDataSet(
    training_df,
    time_idx              = 'time_idx',
    target                = 'log_cases',
    group_ids             = ['group_id'],
    max_encoder_length    = MAX_ENCODER_LEN,
    max_prediction_length = MAX_PREDICTION_LEN,
    static_categoricals   = ['district_enc', 'state_enc', 'disease_enc'],
    static_reals          = ['centroid_lat', 'centroid_lon'],
    time_varying_known_reals    = ['time_idx'],
    time_varying_unknown_reals  = ['log_cases'],
    target_normalizer     = GroupNormalizer(groups=['group_id'], transformation='log1p'),
    allow_missing_timesteps = True,
)

validation = TimeSeriesDataSet.from_dataset(
    training, val_df, predict=True, stop_randomization=True
)

train_loader = training.to_dataloader(train=True,  batch_size=BATCH_SIZE, num_workers=2)
val_loader   = validation.to_dataloader(train=False, batch_size=BATCH_SIZE * 2, num_workers=2)

print(f'Training samples: {len(training)}')
print(f'Validation samples: {len(validation)}')

## 5. Define + train TFT

In [ ]:
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger

tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate      = LEARNING_RATE,
    hidden_size        = HIDDEN_SIZE,
    attention_head_size = ATTENTION_HEADS,
    dropout            = DROPOUT,
    hidden_continuous_size = 16,
    output_size        = 7,   # 7 quantiles: [0.02, 0.1, 0.25, 0.5, 0.75, 0.9, 0.98]
    loss               = QuantileLoss(),
    log_interval       = 10,
    reduce_on_plateau_patience = 5,
)

total_params = sum(p.numel() for p in tft.parameters())
print(f'TFT params: {total_params:,}')

early_stop = EarlyStopping(monitor='val_loss', patience=10, mode='min', verbose=True)
checkpoint = ModelCheckpoint(
    dirpath=DRIVE_OUTPUT,
    filename='tft_best',
    monitor='val_loss',
    mode='min',
    save_top_k=1
)

trainer = pl.Trainer(
    max_epochs        = MAX_EPOCHS,
    accelerator       = 'gpu' if DEVICE == 'cuda' else 'cpu',
    devices           = 1,
    gradient_clip_val = 0.1,
    callbacks         = [early_stop, checkpoint],
    enable_progress_bar = True,
    logger            = TensorBoardLogger(DRIVE_OUTPUT + 'tb_logs', name='tft'),
)

trainer.fit(tft, train_dataloaders=train_loader, val_dataloaders=val_loader)
print(f'Best model: {checkpoint.best_model_path}')
print(f'Best val loss: {checkpoint.best_model_score:.4f}')

## 6. Evaluate — predictions + actuals

In [ ]:
import matplotlib.pyplot as plt

best_tft = TemporalFusionTransformer.load_from_checkpoint(checkpoint.best_model_path)
best_tft.eval()

# Generate predictions on validation set
predictions = best_tft.predict(val_loader, mode='raw', return_x=True)

# Plot a few example series
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for i, ax in enumerate(axes.flat):
    best_tft.plot_prediction(
        predictions.x, predictions.output,
        idx=i * (len(validation) // 4),
        ax=ax, show_future_observed=True
    )
    ax.set_title(f'Sample {i+1}')
plt.tight_layout()
plt.savefig(DRIVE_OUTPUT + 'tft_predictions.png')
plt.show()

# Quantile loss per disease
actuals     = torch.cat([y[0] for x, y in val_loader])
q50_preds   = predictions.output.prediction[:, :, 3]  # median (q50)
mae         = (actuals.cpu() - q50_preds.cpu()).abs().mean()
print(f'MAE (log_cases, q50 median): {mae:.4f}')

## 7. Save TFT artifact

In [ ]:
import torch
from datetime import datetime

artifact = {
    'model_checkpoint_path': checkpoint.best_model_path,
    'dataset_parameters':    training.get_parameters(),
    'training_config': {
        'max_encoder_length':    MAX_ENCODER_LEN,
        'max_prediction_length': MAX_PREDICTION_LEN,
        'diseases':              DISEASES,
        'hidden_size':           HIDDEN_SIZE,
        'attention_heads':       ATTENTION_HEADS,
    },
    'label_encoders': {
        'district': le_district,
        'state':    le_state,
        'disease':  le_disease,
    },
    'best_val_loss': float(checkpoint.best_model_score),
    'trained_at':    datetime.now().isoformat(),
}

save_path = DRIVE_OUTPUT + 'tft_latest.pt'
torch.save(artifact, save_path)
print(f'TFT artifact saved to: {save_path}')
print('Copy the checkpoint .ckpt file AND tft_latest.pt to ml/artifacts/ in your local repo')

from google.colab import files
files.download(save_path)
files.download(checkpoint.best_model_path)
print('Downloads started.')